# 📊 Notebook 2 : Tests d'Hypothèses Statistiques
**Projet :** Prédiction de l'Attrition des Employés — Data Mining  
**Objectif :** Formuler et valider 4 hypothèses statistiques formalisées au seuil $\alpha = 0.05$, avec calcul de la taille d'effet (Cramér's V, $r$ de rang) et correction de tests multiples de Benjamini-Hochberg (FDR).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/raw/Employee_Attrition_DataSet.csv")
print("Dataset chargé :", df.shape)

## Définition des Fonctions d'Effet de Taille

In [ ]:
def cramers_v(contingency_table):
    """Calcul du V de Cramér pour tableau de contingence."""
    chi2 = stats.chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * min(r - 1, k - 1)))

def rank_biserial(group0, group1):
    """Calcul de la corrélation bisérielle de rang (taille d'effet Mann-Whitney)."""
    u_stat, _ = stats.mannwhitneyu(group0, group1)
    n0, n1 = len(group0), len(group1)
    return 1 - (2 * u_stat) / (n0 * n1)

## Hypothèse 1 : OverTime et Attrition (Test du Chi-carré $\chi^2$)
* $H_0$ : La réalisation d'heures supplémentaires est indépendante de l'attrition.
* $H_1$ : Il existe une association significative entre les heures supplémentaires et l'attrition.

In [ ]:
ct_ot = pd.crosstab(df["OverTime"], df["Attrition"])
chi2_ot, p_ot, dof_ot, _ = stats.chi2_contingency(ct_ot)
v_ot = cramers_v(ct_ot)

print(ct_ot)
print(f"\nChi2 = {chi2_ot:.4f}, ddl = {dof_ot}, p-valeur = {p_ot:.4e}")
print(f"Taille d'effet (V de Cramér) = {v_ot:.4f}")
print("Décision :", "Rejet de H0 (Association significative)" if p_ot < 0.05 else "Non-rejet de H0")

## Hypothèse 2 : Revenu Mensuel et Attrition (Test U de Mann-Whitney)
* $H_0$ : La distribution du revenu mensuel est identique entre les employés qui restent et ceux qui partent.
* $H_1$ : Les employés qui partent ont une rémunération mensuelle significativement différente.

In [ ]:
income_no = df[df["Attrition"] == "No"]["MonthlyIncome"]
income_yes = df[df["Attrition"] == "Yes"]["MonthlyIncome"]

# Test de normalité (Shapiro-Wilk sur échantillon de 500)
shapiro_p = stats.shapiro(income_no.sample(500, random_state=42))[1]
print(f"Test de normalité Shapiro-Wilk (p-valeur) : {shapiro_p:.4e} -> Données non-normales.")

# Test non paramétrique de Mann-Whitney U
u_stat, p_mw = stats.mannwhitneyu(income_no, income_yes, alternative="two-sided")
r_effect = rank_biserial(income_no, income_yes)

print(f"Statistique U = {u_stat:.2f}, p-valeur = {p_mw:.4e}")
print(f"Taille d'effet r bisérial de rang = {r_effect:.4f}")
print("Décision :", "Rejet de H0 (Différence significative de salaire)" if p_mw < 0.05 else "Non-rejet de H0")

## Hypothèse 3 : Équilibre Vie Pro/Perso et Attrition (Test du Chi-carré $\chi^2$)
* $H_0$ : Le niveau d'équilibre de vie est indépendant de l'attrition.
* $H_1$ : Il existe une relation significative entre WorkLifeBalance et l'attrition.

In [ ]:
ct_wlb = pd.crosstab(df["WorkLifeBalance"], df["Attrition"])
chi2_wlb, p_wlb, dof_wlb, _ = stats.chi2_contingency(ct_wlb)
v_wlb = cramers_v(ct_wlb)

print(ct_wlb)
print(f"\nChi2 = {chi2_wlb:.4f}, ddl = {dof_wlb}, p-valeur = {p_wlb:.4e}")
print(f"Taille d'effet (V de Cramér) = {v_wlb:.4f}")
print("Décision :", "Rejet de H0" if p_wlb < 0.05 else "Non-rejet de H0")

## Hypothèse 4 : Corrélation Âge vs Ancienneté (Corrélation de Spearman)
* $H_0$ : Il n'y a pas de corrélation monotone entre l'âge et les années d'ancienneté.
* $H_1$ : Il existe une corrélation monotone positive significative.

In [ ]:
rho, p_rho = stats.spearmanr(df["Age"], df["YearsAtCompany"])
print(f"Coefficient rho de Spearman = {rho:.4f}, p-valeur = {p_rho:.4e}")
print("Décision :", "Rejet de H0 (Corrélation positive significative)" if p_rho < 0.05 else "Non-rejet de H0")

## Correction de Tests Multiples (Benjamini-Hochberg FDR)

In [ ]:
# Agrégation des p-valeurs des 4 tests
raw_pvalues = [p_ot, p_mw, p_wlb, p_rho]
tests_labels = [
    "H1: OverTime vs Attrition (Chi2)",
    "H2: MonthlyIncome vs Attrition (Mann-Whitney)",
    "H3: WorkLifeBalance vs Attrition (Chi2)",
    "H4: Age vs YearsAtCompany (Spearman)"
]

reject, p_adjusted, _, _ = multipletests(raw_pvalues, alpha=0.05, method="fdr_bh")

results_table = pd.DataFrame({
    "Hypothèse": tests_labels,
    "p-valeur brute": [f"{p:.4e}" for p in raw_pvalues],
    "p-valeur corrigée (FDR)": [f"{p:.4e}" for p in p_adjusted],
    "Rejet H0 (alpha=0.05)": ["Oui" if r else "Non" for r in reject]
})
results_table